In [ ]:
import scanpy as sc

In [ ]:
sc.settings.set_figure_params(dpi_save=300)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def set_plot_style():
    plt.rcParams.update({
        "font.family": "Arial",
        "axes.linewidth": 1.2,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.edgecolor": "black",
        "grid.color": "#cccccc",
        "grid.linestyle": "--",
        "grid.linewidth": 0.6,
        "axes.grid": True,
        "axes.grid.axis": "y",
        "axes.labelsize": 12,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
    })

set_plot_style()


In [ ]:
mapping = {
    "Basal": "Epithelial",
    "LumA": "Epithelial",
    "LumB": "Epithelial",
    "Normal epithelial": "Epithelial",

    "CAFs myCAF-like": "Fibroblast",
    "CAFs MSC iCAF-like": "Fibroblast",

    "Macrophage M1": "Myeloid",
    "Macrophage M2": "Myeloid",
    "Monocytes": "Myeloid",
    "MDSC": "Myeloid",
    "cDC": "Myeloid",
    "pDC": "Myeloid",

    "Activated CD8": "T Cell",
    "Dysfunc CD8": "T Cell",
    "Terminal exhaustion": "T Cell",
    "Tem": "T Cell",
    "Tcm": "T Cell",
    "Tfh": "T Cell",
    "Deletional tolerance CD8": "T Cell",
    "Precursor exhaustion": "T Cell",
    "Treg": "T Cell",
    "NK": "NK Cell",

    "B cells naive": "B Cell",
    "B cells memory": "B Cell",
    "MBC": "B Cell",
    "Plasmablasts": "B Cell",

    "Endothelial": "Endothelial",

    "PVL immature": "PVL",
    "PVL differentiated": "PVL",
}


In [ ]:
import scanpy as sc
import os

valid_adata_vars = []

files = [f for f in os.listdir('.') if f.startswith('adata_cell_cell_ready_') and f.endswith('.h5ad')]

for file in files:
    sample_id = file.split('ready_')[-1].replace('.h5ad', '')
    final_var = f"adata_{sample_id}"
    print(f"Reading {file} into variable '{final_var}'")

    adata = sc.read(file)

    missing_genes = [gene for gene in ['PLAU', 'PLAUR'] if gene not in adata.var_names]
    if missing_genes:
        print(f"Missing genes in {sample_id}: {', '.join(missing_genes)}")
    else:
        globals()[final_var] = adata  # Save variable globally
        valid_adata_vars.append(final_var)  # Track its name


In [ ]:
for var_name in valid_adata_vars:
    adata = globals()[var_name]
    print(f"Assigning Cell Type for {var_name}")

    adata.obs["Cell Type"] = (
        adata.obs["most_likely_cell_type"]
        .map(mapping)
        .fillna("Other")
    )

In [ ]:
import numpy as np

for var_name in valid_adata_vars:
    adata = globals()[var_name]
    print(f"Processing PLAU_Status for {var_name}")

    plau = adata[:, "PLAU"].X
    if hasattr(plau, "toarray"):
        plau = plau.toarray().ravel()
    else:
        plau = plau.ravel()

    thr_85 = np.quantile(plau, 0.95)

    # Initialize column as "Rest"
    adata.obs["PLAU Status"] = "Rest"

    # Label top 15% as "PLAU_high"
    adata.obs.loc[plau >= thr_85, "PLAU Status"] = "PLAU+"

    # Optional: quick sanity check
    print(adata.obs["PLAU Status"].value_counts(), "\n")

In [ ]:
###Fig8B
sc.pl.spatial(adata, color = ['PLAU Status'], size = 1.5, vmin = 0.5)

In [ ]:
import pandas as pd

all_counts = []

for var_name in valid_adata_vars:
    adata = globals()[var_name]

    if "PLAU Status" not in adata.obs.columns or "Cell Type" not in adata.obs.columns:
        continue

    plau_pos = adata.obs[adata.obs["PLAU Status"] == "PLAU+"]
    ct_counts = plau_pos["Cell Type"].value_counts()
    df_tmp = ct_counts.reset_index()
    df_tmp.columns = ["Cell Type", "Count"]
    df_tmp["Sample"] = var_name

    all_counts.append(df_tmp)

combined_df = pd.concat(all_counts, ignore_index=True)

summary_df = (
    combined_df
    .groupby("Cell Type")["Count"]
    .sum()
    .reset_index()
    .sort_values("Count", ascending=False)
)

summary_df

In [ ]:
import matplotlib.pyplot as plt

df_plot = summary_df.copy()

# Filter for counts > 5
df_plot = df_plot[df_plot["Count"] > 10]

# Sort by count
df_plot = df_plot.sort_values("Count", ascending=False)

plt.figure()
plt.barh(df_plot["Cell Type"], df_plot["Count"], color = 'darkred')
plt.xlabel("Count")
plt.ylabel("")
plt.title("PLAU+ Host Cell TYpes")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.grid(False)
#plt.savefig('Fig8C.pdf', dpi =300)
plt.show()

In [ ]:
###Fig8A
sc.pl.spatial(adata, color=["PLAU"], size=1.5, cmap="inferno", vmin = 0)

In [ ]:
coords = adata.obsm["spatial"]

In [ ]:
from sklearn.neighbors import KDTree
import numpy as np

plau_high_mask = adata.obs["PLAU Status"] == "PLAU+"
plau_coords = coords[plau_high_mask.values]

tree = KDTree(plau_coords)

In [ ]:
d = adata.obs["dist_PLAU_high"]
plau_high_mask = adata.obs["PLAU Status"] == "PLAU+"

r1 = 400
r2 = 700
r3 = 1200

adata.obs["PLAU_ring"] = "Outer"

adata.obs.loc[plau_high_mask, "PLAU_ring"] = "PLAU+ Host"

adata.obs.loc[(~plau_high_mask) & (d <= r1), "PLAU_ring"] = "Ring1"
adata.obs.loc[(~plau_high_mask) & (d > r1) & (d <= r2), "PLAU_ring"] = "Ring2"
adata.obs.loc[(~plau_high_mask) & (d > r2) & (d <= r3), "PLAU_ring"] = "Ring3"

In [ ]:
adata.obs["PLAU_zone"] = "PLAU-Far"

adata.obs.loc[
    adata_092A.obs["PLAU_ring"].isin(["Ring1", "Ring2"]),
    "PLAU_zone"
] = "PLAU-Near"

adata.obs.loc[
    adata_092A.obs["PLAU_ring"] == "PLAU+ Host",
    "PLAU_zone"
] = "PLAU+ Host"

In [ ]:
###Fig8D
sc.pl.spatial(adata, color = ['PLAU_zone'], size = 1.5, vmin = 0.5)

In [ ]:
emt_genes = ["VIM", "FN1", "SNAI1", "SNAI2", "TWIST1", "ZEB1", "ZEB2", "ITGA5", "ITGB1", "CDH2", "CD44", "COL1A1", "COL1A2", "SERPINE1", "MMP2", "MMP9", "SOX4", "SOX9", "TGFBI"]

In [ ]:
from sklearn.neighbors import KDTree
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu
import scanpy as sc

results = []

for var_name in valid_adata_vars:

    adata = globals()[var_name]
    print(f"Processing {var_name}")

    coords = adata.obsm["spatial"]

    plau_high_mask = adata.obs["PLAU Status"] == "PLAU+"

    if plau_high_mask.sum() < 5:
        print("Skipping: too few PLAU+ spots")
        continue

    plau_coords = coords[plau_high_mask.values]
    tree = KDTree(plau_coords)

    distances, _ = tree.query(coords, k=1)
    adata.obs["dist_PLAU_high"] = distances.ravel()

    d = adata.obs["dist_PLAU_high"]

    r1, r2, r3 = np.quantile(d, [0.25, 0.5, 0.75])

    adata.obs["PLAU_ring"] = "Outer"
    adata.obs.loc[plau_high_mask, "PLAU_ring"] = "PLAU+ Host"

    adata.obs.loc[(~plau_high_mask) & (d <= r1), "PLAU_ring"] = "Ring1"
    adata.obs.loc[(~plau_high_mask) & (d > r1) & (d <= r2), "PLAU_ring"] = "Ring2"
    adata.obs.loc[(~plau_high_mask) & (d > r2) & (d <= r3), "PLAU_ring"] = "Ring3"

    adata.obs["PLAU_zone"] = "PLAU-Far"

    adata.obs.loc[
        adata.obs["PLAU_ring"].isin(["Ring1", "Ring2"]),
        "PLAU_zone"
    ] = "PLAU-Near"

    adata.obs.loc[
        adata.obs["PLAU_ring"] == "PLAU+ Host",
        "PLAU_zone"
    ] = "PLAU+ Host"

    epi_mask = adata.obs["Cell Type"] == "Epithelial"

    n_near = np.sum(epi_mask & (adata.obs["PLAU_zone"] == "PLAU-Near"))
    n_far  = np.sum(epi_mask & (adata.obs["PLAU_zone"] == "PLAU-Far"))

    sc.tl.score_genes(
        adata,
        gene_list=emt_genes,
        score_name="EMT_Score",
        use_raw=False
    )

    emt_near = adata.obs.loc[
        epi_mask & (adata.obs["PLAU_zone"] == "PLAU-Near"),
        "EMT_Score"
    ]

    emt_far = adata.obs.loc[
        epi_mask & (adata.obs["PLAU_zone"] == "PLAU-Far"),
        "EMT_Score"
    ]

    if len(emt_near) > 10 and len(emt_far) > 10:
        stat, pval_emt = mannwhitneyu(
            emt_near, emt_far, alternative="greater"
        )
    else:
        pval_emt = np.nan

    results.append({
        "sample": var_name,
        "n_epi_near": n_near,
        "n_epi_far": n_far,
        "EMT_near_mean": emt_near.median(),
        "EMT_far_mean": emt_far.mean(),
        "EMT_p": pval_emt
    })

results_df = pd.DataFrame(results)
results_df

In [ ]:
results = []

for var_name in valid_adata_vars:

    adata = globals()[var_name]
    print(f"Processing {var_name}")

    coords = adata.obsm["spatial"]

    plau_high_mask = adata.obs["PLAU Status"] == "PLAU+"

    if plau_high_mask.sum() < 5:
        print("Skipping: too few PLAU+ spots")
        continue

    plau_coords = coords[plau_high_mask.values]
    tree = KDTree(plau_coords)

    distances, _ = tree.query(coords, k=1)
    adata.obs["dist_PLAU_high"] = distances.ravel()

    d = adata.obs["dist_PLAU_high"]

    r1, r2, r3 = np.quantile(d, [0.25, 0.5, 0.75])

    adata.obs["PLAU_ring"] = "Outer"
    adata.obs.loc[plau_high_mask, "PLAU_ring"] = "PLAU+ Host"

    adata.obs.loc[(~plau_high_mask) & (d <= r1), "PLAU_ring"] = "Ring1"
    adata.obs.loc[(~plau_high_mask) & (d > r1) & (d <= r2), "PLAU_ring"] = "Ring2"
    adata.obs.loc[(~plau_high_mask) & (d > r2) & (d <= r3), "PLAU_ring"] = "Ring3"

    adata.obs["PLAU_zone"] = "PLAU-Far"

    adata.obs.loc[
        adata.obs["PLAU_ring"].isin(["Ring1", "Ring2"]),
        "PLAU_zone"
    ] = "PLAU-Near"

    adata.obs.loc[
        adata.obs["PLAU_ring"] == "PLAU+ Host",
        "PLAU_zone"
    ] = "PLAU+ Host"

    epi_mask = adata.obs["Cell Type"] == "Epithelial"

    sc.tl.score_genes(
        adata,
        gene_list=msig_EMT_curated,
        score_name="EMT_Score",
        use_raw=False
    )

    emt_near = adata.obs.loc[
        epi_mask & (adata.obs["PLAU_zone"] == "PLAU-Near"),
        "EMT_Score"
    ].dropna()

    emt_far = adata.obs.loc[
        epi_mask & (adata.obs["PLAU_zone"] == "PLAU-Far"),
        "EMT_Score"
    ].dropna()

    emt_all_epi = adata.obs.loc[
        epi_mask & adata.obs["PLAU_zone"].isin(["PLAU-Near", "PLAU-Far"]),
        "EMT_Score"
    ].dropna()

    n_near = len(emt_near)
    n_far = len(emt_far)

    if n_near > 10 and n_far > 10:

        stat, pval_emt = mannwhitneyu(
            emt_near,
            emt_far,
            alternative="greater"
        )

        emt_near_median = emt_near.median()
        emt_far_median = emt_far.median()

        emt_delta = emt_near_median - emt_far_median

        emt_all_median = emt_all_epi.median()

        emt_enrichment_median_ratio = emt_delta / emt_all_median
        emt_all_median = emt_all_epi.median()

        if np.abs(emt_all_median) > 1e-8:
            emt_median_ratio_absdenom = emt_delta / np.abs(emt_all_median)
        else:
            emt_median_ratio_absdenom = np.nan
    else:
        pval_emt = np.nan
        emt_near_median = np.nan
        emt_far_median = np.nan
        emt_delta = np.nan
        emt_all_median = np.nan
        emt_enrichment_z = np.nan

    results.append({
        "sample": var_name,
        "n_epi_near": n_near,
        "n_epi_far": n_far,
        "EMT_near_median": emt_near_median,
        "EMT_far_median": emt_far_median,
        "EMT_delta": emt_delta,
        "EMT_near_far_median": emt_all_median,
        "EMT_median_ratio_absdenom": emt_median_ratio_absdenom,
        "EMT_p": pval_emt
    })

results_df = pd.DataFrame(results)
results_df

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon

df_plot = results_df.copy()

df_plot = df_plot.sort_values("EMT_median_ratio_absdenom")

# Clean sample names
def clean_name(s):
    s = s.replace("adata_", "")
    for i, ch in enumerate(s):
        if ch.isalpha():
            return s[:i] + ch.upper() + s[i+1:]
    return s

df_plot["clean_sample"] = df_plot["sample"].apply(clean_name)

rb = df_plot["EMT_median_ratio_absdenom"].dropna()
stat, p_patient = wilcoxon(
    rb,
    alternative="greater"
)
print("Median rank-biserial:", rb.median())
print("Positive samples:", (rb > 0).sum(), "/", len(rb))
print("Patient-level Wilcoxon p:", p_patient)

y = np.arange(len(df_plot))

plt.figure(figsize=(5, 5))

# Color by within-sample Mann-Whitney p-value
line_colors = [
    "darkred" if p < 0.05 else "darkred"
    for p in df_plot["EMT_p"]
]

for i, row in enumerate(df_plot.itertuples()):
    plt.hlines(
        y=i,
        xmin=0,
        xmax=row.EMT_median_ratio_absdenom,
        linewidth=6,
        color=line_colors[i]
    )

plt.scatter(
    df_plot["EMT_median_ratio_absdenom"],
    y,
    c=line_colors,
    s=70,
    zorder=3
)

plt.axvline(0, color="gray", linewidth=1)

for i, row in enumerate(df_plot.itertuples()):
    plt.text(
        row.EMT_median_ratio_absdenom + 0.015,
        i,
        f"p={row.EMT_p:.1g}",
        va="center",
        ha="left",
        fontsize=12
    )

plt.yticks(y, df_plot["clean_sample"], fontsize = 12)
plt.ylabel('Patient Identifier', labelpad=10)
plt.xlabel(
    r"$\frac{(EMT_{\mathrm{Near}}) - (EMT_{\mathrm{Far}})}{(EMT_{\mathrm{Near+Far}})}$",
    fontsize=16,
    labelpad = 10
)
'''
plt.title(
    f"Patient-normalized EMT enrichment near PLAU+ regions\n"
    f"Wilcoxon p={p_patient:.3g}"
)
'''
plt.grid(False)
plt.tight_layout()
plt.savefig("Fig8F.pdf", dpi=300, bbox_inches="tight")
plt.show()